# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omar-mowafy66/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

## My rule and its reason codes

I rank content higher when it has a stronger refresh opportunity and meaningful search volume.

The rule uses two signals:
- Staleness: `days_since_last_update`
- Search volume: `search_volume`

Higher scores indicate higher priority for review.

### Reason codes

- `stale_high_volume`: Content is stale and has meaningful search demand.
- `stale`: Content is stale but has lower search demand.
- `high_volume`: Content has meaningful search demand but is not highly stale.
- `monitor`: Neither signal indicates a strong opportunity.

### Action labels

- `refresh`: Prioritize the content for a refresh review.
- `review`: Review the content for a possible opportunity.
- `monitor`: No immediate action.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [20]:
# Section 2 — Build the ranked queue

import os
import pandas as pd
import numpy as np

# Make sure the dataset is loaded
if "df" not in globals():
    raise NameError(
        "df is not loaded. Run the dataset-loading cell before this cell."
    )

df_score = df.copy()

# --------------------------------------------------
# 1. Prepare the two signals
# --------------------------------------------------

df_score["days_since_last_update"] = pd.to_numeric(
    df_score["days_since_last_update"],
    errors="coerce"
).fillna(0)

df_score["search_volume"] = pd.to_numeric(
    df_score["search_volume"],
    errors="coerce"
).fillna(0)

# --------------------------------------------------
# 2. Normalize signals
# --------------------------------------------------

staleness_max = df_score["days_since_last_update"].max()
volume_max = df_score["search_volume"].max()

if staleness_max > 0:
    df_score["staleness_score"] = (
        df_score["days_since_last_update"] / staleness_max
    )
else:
    df_score["staleness_score"] = 0.0

if volume_max > 0:
    df_score["volume_score"] = (
        df_score["search_volume"] / volume_max
    )
else:
    df_score["volume_score"] = 0.0

# --------------------------------------------------
# 3. Baseline score
# --------------------------------------------------

df_score["score"] = (
    0.6 * df_score["staleness_score"]
    + 0.4 * df_score["volume_score"]
)

# --------------------------------------------------
# 4. Define thresholds
# --------------------------------------------------

volume_threshold = df_score["search_volume"].median()

# --------------------------------------------------
# 5. Reason codes
# --------------------------------------------------

df_score["reason_code"] = np.select(
    [
        (
            (df_score["days_since_last_update"] >= 180)
            & (df_score["search_volume"] >= volume_threshold)
        ),
        (
            df_score["days_since_last_update"] >= 180
        ),
        (
            df_score["search_volume"] >= volume_threshold
        )
    ],
    [
        "stale_high_volume",
        "stale",
        "high_volume"
    ],
    default="monitor"
)

# --------------------------------------------------
# 6. Action labels
# --------------------------------------------------

df_score["action_label"] = np.select(
    [
        df_score["reason_code"] == "stale_high_volume",
        df_score["reason_code"].isin(
            ["stale", "high_volume"]
        )
    ],
    [
        "refresh",
        "review"
    ],
    default="monitor"
)

# --------------------------------------------------
# 7. Rank the queue
# --------------------------------------------------

df_score = (
    df_score
    .sort_values(
        by=["score", "content_id"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

df_score["rank"] = np.arange(1, len(df_score) + 1)

# --------------------------------------------------
# 8. Write the required CSV
# --------------------------------------------------

output_path = "work/outputs/baseline_action_score.csv"

os.makedirs(
    os.path.dirname(output_path),
    exist_ok=True
)

df_score.to_csv(
    output_path,
    index=False
)

print("Baseline completed successfully.")
print(f"Rows: {len(df_score):,}")
print(f"Output: {output_path}")

# --------------------------------------------------
# 9. Display Top 20
# --------------------------------------------------

top20 = df_score.head(20)

display(
    top20[
        [
            "rank",
            "content_id",
            "days_since_last_update",
            "search_volume",
            "score",
            "reason_code",
            "action_label"
        ]
    ]
)

Baseline completed successfully.
Rows: 30,000
Output: work/outputs/baseline_action_score.csv


,rank,content_id,days_since_last_update,search_volume,score,reason_code,action_label
0,1,content_3f3576c295f5,373,0.0,0.600000,stale,review
1,2,content_55a5b1c46474,373,0.0,0.600000,stale,review
2,3,content_f6fdf87348f6,373,0.0,0.600000,stale,review
3,4,content_1b4ec72dafd4,372,0.0,0.598391,stale,review
4,5,content_8d56efff1e71,372,0.0,0.598391,stale,review
5,6,content_ef99c4abd9ab,104,74000.0,0.567292,high_volume,review
6,7,content_f01216059a6a,335,0.0,0.538874,stale,review
7,8,content_06e19c6486b0,334,0.0,0.537265,stale,review
8,9,content_e2b702f4f92b,334,0.0,0.537265,stale,review
9,10,content_02b0d6e30129,313,110.0,0.504080,stale_high_volume,refresh


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [21]:
# Section 3 — Top-20 review

top20 = df_score.head(20).copy()

display(
    top20[
        [
            "rank",
            "content_id",
            "score",
            "reason_code",
            "action_label",
            "days_since_last_update",
            "search_volume"
        ]
    ]
)

,rank,content_id,score,reason_code,action_label,days_since_last_update,search_volume
0,1,content_3f3576c295f5,0.600000,stale,review,373,0.0
1,2,content_55a5b1c46474,0.600000,stale,review,373,0.0
2,3,content_f6fdf87348f6,0.600000,stale,review,373,0.0
3,4,content_1b4ec72dafd4,0.598391,stale,review,372,0.0
4,5,content_8d56efff1e71,0.598391,stale,review,372,0.0
5,6,content_ef99c4abd9ab,0.567292,high_volume,review,104,74000.0
6,7,content_f01216059a6a,0.538874,stale,review,335,0.0
7,8,content_06e19c6486b0,0.537265,stale,review,334,0.0
8,9,content_e2b702f4f92b,0.537265,stale,review,334,0.0
9,10,content_02b0d6e30129,0.504080,stale_high_volume,refresh,313,110.0


## 4. Weak picks + leakage check
## Weak Picks + Leakage Check

### Weak picks

Some of the highest-ranked rows are weak picks because the baseline relies on only two signals.

The stale-only rows with zero search volume are questionable priorities. They have high scores because of their age, but the lack of search demand makes their practical impact uncertain.

The high-volume-only rows are also uncertain because search volume shows demand but does not prove that the content needs an update.

These cases show that the baseline is useful for prioritization but should not be treated as a final decision rule.

### Leakage check

The baseline uses only `days_since_last_update` and `search_volume`.

I did not use `trend_pct` or `trend_direction`, because these signals can contain future or label-derived information.

I also did not use product flags or future-window information.

The resulting queue is therefore intended as a baseline for decision support, not as a predictive model.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.